<a href="https://colab.research.google.com/github/irenehavsa/mental-disorder-prevalence/blob/main/notebooks/00_fetch_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install kagglehub
!pip install -q kagglehub

In [ ]:
# Upload API Key file
from google.colab import files
files.upload()

# Move API Key file to the correct location
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [7]:
# Import required packages
import kagglehub
import os
import shutil
from pathlib import Path
import pandas as pd

###1. Import Raw Data

In [ ]:
# Download latest version of the dataset from Kaggle
path = kagglehub.dataset_download("thedevastator/uncover-global-trends-in-mental-health-disorder")

In [5]:
# Create "data/raw" directory
raw_data_dir = Path("data/raw")
raw_data_dir.mkdir(parents=True, exist_ok=True)

# Define source and destination
source_file = Path(path) / "Mental health Depression disorder Data.csv"
destination_file = raw_data_dir / "mental_health_depression_disorder_data.csv"

# Copy file
shutil.copy(source_file, destination_file)

print("Dataset saved to data/raw/")

Dataset saved to data/raw/


###2. Explore Raw Data

In [8]:
df = pd.read_csv(source_file)
display(df.head())

/tmp/ipython-input-4038649746.py:1: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(source_file)


,index,Entity,Code,Year,Schizophrenia (%),Bipolar disorder (%),Eating disorders (%),Anxiety disorders (%),Drug use disorders (%),Depression (%),Alcohol use disorders (%)
0,0,Afghanistan,AFG,1990,0.16056,0.697779,0.101855,4.828830,1.677082,4.071831,0.672404
1,1,Afghanistan,AFG,1991,0.160312,0.697961,0.099313,4.829740,1.684746,4.079531,0.671768
2,2,Afghanistan,AFG,1992,0.160135,0.698107,0.096692,4.831108,1.694334,4.088358,0.670644
3,3,Afghanistan,AFG,1993,0.160037,0.698257,0.094336,4.830864,1.705320,4.096190,0.669738
4,4,Afghanistan,AFG,1994,0.160022,0.698469,0.092439,4.829423,1.716069,4.099582,0.669260


In [ ]:
# See the number of rows and columns
df.shape

In [ ]:
# See list of columns and their data type
df.info()

# Note of result-1: the columns' naming convention are not standard. I'll modify them into snake_case and remove unnecessary symbols and words.
# Note of result-2: the original index column is not needed for analysis, thus should be removed.

In [ ]:
# See list of countries
df.Entity.unique()

# Note of result-1: There is an entity "World", which is the prevalance data of the whole countries combined. There is also "Southeast Asia", which should represent the prevalance of all SEA countries combined.
# Note of result-2: Timor separated from Indonesia in May 2002, therefore if there is prevalence data before that year, it is unreliable. Note that this data is synthethic and there could be parts that don't make sense.

In [ ]:
# See list of years
df.Year.unique()

# Note of result: there is data of the BCE years, which shouldn't be possible. I'll explore the data on the next step.

In [ ]:
# Explore data of various years and value of columns

# df.loc[(df['Year'] == '1000 BCE')]
# df.loc[(df['Year'] == '1990')]
# df.loc[(df['Depression (%)'].notnull())]
df.loc[(df['Year'] == '1990') & (df['Depression (%)'].notnull())]

# Note of result: only data with the year 1990 and later AND depression data not null is valid, therefore the other should be removed.

In [ ]:
# See sample of complete data of a country
df.loc[(df['Entity'] == 'Indonesia') & (df['Depression (%)'].notnull())]

# Note of result: here it shows that there is data for Timor for the years before 2002. They are not valid since at those time Timor was still part of Indonesia. Thus, the data will be removed.

###3. Cleanse and Prepare Data for Analysis

From the original data exploration above, here is what I'll do:
1.   Remove 'index' column
2.   Rename columns' name into snake_case and remove unnecessary symbols and words to make them compact
3.   Since the scope of the analysis is Southeast Asia (SEA), only include data with Entity includes SEA countries, "Southeast Asia", and "World"
4.   Only include data with the year 1990 or later and with 'depression' column not null.
5.   Exclude data of East Timor ("Timor") before the year 2002, since it is not valid   

In [11]:
# Remove the original index column since it will not be used in this analysis
df = df.drop(columns=['index'])

# Rename columns' name
df.columns = ['entity', 'code', 'year', 'schizophrenia', 'bipolar', 'eating_disorder', 'anxiety', 'drug_use', 'depression', 'alcohol_use']

# Convert prevalence columns to numeric, coercing errors to NaN
prevalence_columns = ['schizophrenia', 'bipolar', 'eating_disorder', 'anxiety', 'drug_use', 'depression', 'alcohol_use']
for col in prevalence_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Include only relevant entities
entities_to_include = ['Southeast Asia', 'World', 'Indonesia', 'Timor', 'Malaysia', 'Singapore', 'Brunei', 'Philippines', 'Myanmar', 'Thailand', 'Laos', 'Cambodia']
df = df[df['entity'].isin(entities_to_include)]

# Include only valid data: year >= 1990 and 'depression' is not null
# First change the 'year' data type into numeric
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df = df[(df['year'] >= 1990) & (df['depression'].notnull())]

# Apply country-specific minimum year thresholds
df = df[(df['entity'] != 'Timor') | (df['year'] >= 2002)]

# Display the updated dataframe
display(df.head())

,entity,code,year,schizophrenia,bipolar,eating_disorder,anxiety,drug_use,depression,alcohol_use
784,Brunei,BRN,1990.0,0.274544,0.617719,0.550731,3.548453,0.911231,2.577977,0.771017
785,Brunei,BRN,1991.0,0.273819,0.618347,0.548010,3.554565,0.910156,2.562828,0.770862
786,Brunei,BRN,1992.0,0.273212,0.618986,0.545544,3.561621,0.910287,2.551382,0.770430
787,Brunei,BRN,1993.0,0.272736,0.619673,0.543644,3.569155,0.910409,2.543804,0.769584
788,Brunei,BRN,1994.0,0.272383,0.620342,0.542626,3.576319,0.908915,2.538465,0.768343


In [ ]:
# There should be 11*28 + 1*16 (for Timor) = 324 rows
df.shape

In [ ]:
# Review updated dataframe
# df.loc[(df['entity'] == 'Southeast Asia')]
df.loc[(df['entity'] == 'Thailand')]

In [12]:
# 'Unpivot' prevalence data
df = pd.melt(df, id_vars=['entity', 'code', 'year'], var_name='mental_disorder', value_name='prevalence')
df.head()

,entity,code,year,mental_disorder,prevalence
0,Brunei,BRN,1990.0,schizophrenia,0.274544
1,Brunei,BRN,1991.0,schizophrenia,0.273819
2,Brunei,BRN,1992.0,schizophrenia,0.273212
3,Brunei,BRN,1993.0,schizophrenia,0.272736
4,Brunei,BRN,1994.0,schizophrenia,0.272383


In [15]:
# Create "data/processed" directory
raw_data_dir = Path("data/processed")
raw_data_dir.mkdir(parents=True, exist_ok=True)

# Save as a new CSV file
df.to_csv(
    "data/processed/mental_disorders_prevalence_yearly.csv",
    index=False
)

print("Dataset saved to data/processed/")